# N104 · 序列匹配、编辑距离与子序列计数

**目标：** 按两个前缀定义状态并区分最优值和方案数。

**先修：** N098, N022。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** LCS；编辑增删改；不同子序列；空串边界；路径恢复。

**配套讲解来源：** [同名逐章意见](../../comment/104_sequence_alignment_dp.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

给定两个字符串（或序列），本章解决三个"对齐比较"问题。它们的共同点是把状态定义在"两个前缀"上，画成一张二维网格；不同点在于网格里填的数字和合并方式完全不同。

1. **最长公共子序列（LCS）**：在两个字符串里各挑一个子序列（保持顺序、可不连续），让它们完全相等，问最长能有多长。比如 `a='abcde'`、`b='ace'`，答案是 `3`：公共子序列 `'ace'` 长度为 3，没有更长的了。
2. **编辑距离**：把字符串 a 改成 b 最少需要几步操作，每步可以插入一个字符、删除一个字符或替换一个字符。比如 `'horse'` 改成 `'ros'` 是 `3` 步：horse → rorse（把首位的 h 替换成 r）→ rose（删掉第 2 位的 r）→ ros（删掉末尾的 e），三步完成，且没有两步的方案。
3. **不同子序列（计数）**：s 的子序列里有多少个恰好等于 t。比如 `s='rabbbit'`、`t='rabbit'`，答案是 `3`：三只 b 里任选两只配成 "bb"，共 3 种选法。再如 `s='abc'`、`t=''`，答案是 `1`：空串只能由"什么都不选"这一种方式构成。

三个问题一张网格，但格子里分别做 max（求最长）、min（求最省）、加法（数方案），这就是 “思路推导”部分 说的"共享网格形状、不同代数"。

## 2. 基础知识：先读懂这些词

- **二维前缀状态 `dp[i][j]`**：把"前 i 个 a 元素"和"前 j 个 b 元素"这两段前缀配成对，dp[i][j] 记录这对前缀上的答案。i、j 各自增长，形成 (m+1)×(n+1) 的表格，第 0 行第 0 列天然表示"空前缀"。
- **子序列**：保持相对顺序、允许跳过字符的挑选结果。'ace' 是 'abcde' 的子序列（跳过了 b、d）。
- **LCS（最长公共子序列）**：同时是两个序列子序列的最长串。diff 工具、版本比较的底层就是它。
- **编辑距离（Levenshtein 距离）**：插入/删除/替换单位成本都算 1 步（“正确性与复杂度”部分 特别注明本章是"单位成本"版本）。
- **对齐最后一项**：LCS 转移的思考方式——看两个前缀的最后一个字符，若相等，可以让它们配成一对（对齐），两边同时退一格、长度加一；若不等，至少有一个末字符不进公共子序列，丢掉一端。
- **按最后一次操作分解**：编辑距离的思考方式——把 a 的前 i 个改成 b 的前 j 个，最后一步操作只能是三种之一：删 a 的末字符、插入 b 的末字符、替换（对齐）两个末字符。三种来源取最省。
- **计数 DP 与"用不用当前字符"**：num_distinct 的状态 dp[j] 是"s 的前缀里匹配 t 前 j 个字符的方案数"。转移按"s 的当前字符是否被用于匹配"分两批相加。它是加法（sum）语义，与 max/min 完全不同，混用就是 bug。
- **空串边界**：dp 表第 0 行第 0 列必须显式定义。LCS：任一前缀为空则公共长度 0；编辑距离：空串改成 b 前 j 个要插 j 次（所以初始化是 0,1,2,...j）；计数：匹配空 t 恰好 1 种方案（什么都不选），其余为 0。
- **滚动数组与逆序更新**：三个实现都把二维表压成一行。lcs 和 edit_distance 用"old 行 + 新行"双数组避免覆盖；num_distinct 用单数组**从右往左**更新，道理与 0/1 背包逆序相同：保证读到的 `dp[j-1]` 还是上一轮（上一个字符处理前）的值，一个 s 字符不会被重复用于匹配 t 的多个位置。

## 3. 思路推导：从小例子开始

**问题 1：LCS**

- Step 1：定义 `dp[i][j]` = a 前 i 个与 b 前 j 个的 LCS 长度。第 0 行第 0 列全 0（空前缀没有公共内容）。
- Step 2：看末字符 a[i−1] 与 b[j−1]。相等时，存在一个最优解把它们配成一对：`dp[i][j]=dp[i-1][j-1]+1`；不等时，末字符至少有一个要被丢弃：`dp[i][j]=max(dp[i-1][j], dp[i][j-1])`。
- Step 3：填完整张表，右下角 `dp[m][n]` 即答案。

“运行示例”部分 演示了 `a='abcde'`、`b='ace'` 的整张表（每行是"a 的前缀固定后，b 各前缀的 LCS 长度"）：

| a 的前缀 | b 前缀的 LCS 长度（j=0..3） |
|---|---|
| ''（空前缀） | 0, 0, 0, 0 |
| 'a' | 0, 1, 1, 1 |
| 'ab' | 0, 1, 1, 1 |
| 'abc' | 0, 1, 2, 2 |
| 'abcd' | 0, 1, 2, 2 |
| 'abcde' | 0, 1, 2, 3 |

手看两个关键格子：行 `'abc'`、列 j=2（b 前缀 'ac'），末字符 c==c 相等，得 dp[2][1]+1=2；最后一行 j=3，e==e 相等，得 dp['abcd'][j=2]+1=3。右下角 3 就是 LCS 长度，对应 'ace'。

**问题 2：编辑距离**

- Step 1：`dp[i][j]` = a 前 i 个改成 b 前 j 个的最少操作数。第 0 行初始化为 0,1,...,j（纯插入），第 0 列每行是 0,1,...,i（纯删除）。
- Step 2：末字符相等时，最省的做法是对齐它们、剩下的问题照旧：`dp[i][j]=dp[i-1][j-1]`。不等时枚举最后一步操作的三种可能：删除 a 末字符（`1+old[j]`，即 dp[i−1][j]+1）、在结尾插入 b 末字符（`1+dp[j-1]`，即 dp[i][j−1]+1）、替换末字符使其对齐（`1+old[j-1]`，即 dp[i−1][j−1]+1），三者取最小。

手算 `a='ab'`、`b='a'`：初始行（空前缀）为 [0,1]；处理 'a'：末字符相等，dp[1]=old[0]=0，此行 [1,0]；处理 'b'：与 'a' 不等，dp[1]=1+min(old[1]=0, dp[0]=2, old[0]=1)=1。答案 1——删掉 'b' 即可。`'horse'`→`'ros'` 同理逐格填满得 3。

**问题 3：不同子序列计数**

- Step 1：`dp[j]` = 目前扫过的 s 前缀里，匹配 t 前 j 个字符的方案数。初始 `dp=[1]+[0]*len(t)`：匹配空 t 有 1 种（什么都不选）。
- Step 2：每读入 s 的一个字符 x，从右往左扫 j：若 x 等于 t[j−1]，则新方案多出一批"用 x 充当 t 第 j 个字符"的，数量等于旧 dp[j−1]，做加法；不等则不产生新方案。右往左保证了 dp[j−1] 是"还没算 x 影响"的旧值。
- Step 3：处理完 s，dp[-1] 就是方案数。

手算 `s='aab'`、`t='ab'`：初始 [1,0,0]；第一个 'a'：j=1 命中，dp=[1,1,0]；第二个 'a'：j=1 又命中，dp=[1,2,0]（第一个 a 匹配或第二个 a 匹配，两种）；'b'：j=2 命中，dp[2]+=dp[1] 得 [1,2,2]。答案 2，对应用第一或第二个 'a' 两种选法。

## 4. 核心代码：lcs

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def lcs(a, b):
    dp = [0] * (len(b) + 1)
    for x in a:
        old = dp
        dp = [0] * (len(b) + 1)
        for j, y in enumerate(b, 1):
            dp[j] = old[j - 1] + 1 if x == y else max(old[j], dp[j - 1])
    return dp[-1]
```

### 逐段读懂代码

**函数一：`lcs`**

```python
def lcs(a, b):
    dp = [0] * (len(b) + 1)
    for x in a:
        old = dp
        dp = [0] * (len(b) + 1)
        for j, y in enumerate(b, 1):
            dp[j] = old[j - 1] + 1 if x == y else max(old[j], dp[j - 1])
    return dp[-1]
```

- `dp=[0]*(len(b)+1)`：对应二维表的第 0 行，空前缀与任何 b 前缀的 LCS 都是 0。
- `old=dp; dp=[0]*(len(b)+1)`：滚动到新行。`old` 保管上一行（a 短一格的答案），`dp` 是新行且第 0 列天然是 0。
- `enumerate(b,1)`：下标从 1 开始，正好对应"前 j 个字符"的 j。
- `old[j-1]+1 if x==y else max(old[j],dp[j-1])`：三元表达式紧凑地写出两种情况——末字符相等就左上角加一；不等就取上方（`old[j]`，丢 a 末字符）与左方（`dp[j-1]`，丢 b 末字符）的较大者。
- `return dp[-1]`：最后一行最后一列。

**函数二：`edit_distance`**

```python
def edit_distance(a, b):
    dp = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        old = dp
        dp = [i] + [0] * len(b)
        for j, y in enumerate(b, 1):
            dp[j] = old[j - 1] if x == y else 1 + min(old[j], dp[j - 1], old[j - 1])
    return dp[-1]
```

- `dp=list(range(len(b)+1))`：第 0 行是 0,1,2,...,n——空前缀改成 b 前 j 个字符要插 j 次。
- `dp=[i]+[0]*len(b)`：新行第 0 列是 i，即 a 前 i 个改成空串要删 i 次。
- 相等分支 `old[j-1]`：对齐末字符，不花操作。
- 不等分支 `1+min(old[j],dp[j-1],old[j-1])`：三个来源分别是"删 a 末字符"（上一行同列）、"末尾插入 b 末字符"（本行左列）、"替换对齐"（左上角），各花 1 步，取最省。

**函数三：`num_distinct`**

```python
def num_distinct(s, t):
    dp = [1] + [0] * len(t)
    for x in s:
        for j in range(len(t), 0, -1):
            if x == t[j - 1]:
                dp[j] += dp[j - 1]
    return dp[-1]
```

- `dp=[1]+[0]*len(t)`：dp[0]=1 是"匹配空 t 的方案数"，也是整个计数的地基。
- `for j in range(len(t),0,-1)`：从 len(t) 倒着扫到 1。倒序是关键——dp[j] 更新时要加的 dp[j−1] 必须是"还没被本轮 x 更新过"的旧值；如果正序扫，dp[j−1] 已经掺入 x 的贡献，等于同一个 x 被用了两次。
- `if x==t[j-1]: dp[j]+=dp[j-1]`：x 不匹配时 dp[j] 原样保留（"不用 x"的那批方案本来就已在里面），匹配时叠加"用 x 匹配 t 第 j 位"的那批。
- `return dp[-1]`：完整 t 的匹配方案数。

## 5. 分段实现：按顺序运行

**本章接口：** `lcs(a, b)`、`edit_distance(a, b)`、`num_distinct(s, t)`

### lcs

In [1]:
def lcs(a, b):
    dp = [0] * (len(b) + 1)
    for x in a:
        old = dp
        dp = [0] * (len(b) + 1)
        for j, y in enumerate(b, 1):
            dp[j] = old[j - 1] + 1 if x == y else max(old[j], dp[j - 1])
    return dp[-1]

### edit_distance

In [2]:
def edit_distance(a, b):
    dp = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        old = dp
        dp = [i] + [0] * len(b)
        for j, y in enumerate(b, 1):
            dp[j] = old[j - 1] if x == y else 1 + min(old[j], dp[j - 1], old[j - 1])
    return dp[-1]

### num_distinct

In [3]:
def num_distinct(s, t):
    dp = [1] + [0] * len(t)
    for x in s:
        for j in range(len(t), 0, -1):
            if x == t[j - 1]:
                dp[j] += dp[j - 1]
    return dp[-1]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a,b='abcde','ace'; dp=[[0]*(len(b)+1) for _ in range(len(a)+1)]
for i,x in enumerate(a,1):
    for j,y in enumerate(b,1): dp[i][j]=dp[i-1][j-1]+1 if x==y else max(dp[i-1][j],dp[i][j-1])
show_table(['a的前缀','b前缀的LCS长度'],[(a[:i],row) for i,row in enumerate(dp)])

a的前缀,b前缀的LCS长度
,"[0, 0, 0, 0]"
a,"[0, 1, 1, 1]"
ab,"[0, 1, 1, 1]"
abc,"[0, 1, 2, 2]"
abcd,"[0, 1, 2, 2]"
abcde,"[0, 1, 2, 3]"


## 7. 正确性、适用条件与复杂度

前缀边界明确处理空序列。LCS相同末字符可以在某个最优解中对齐，不同末字符至少丢弃一端；编辑距离按最后操作分解；计数的逆序更新保证一个来源字符至多匹配目标中的一个位置。

**代价：** 各算法O(mn)算术操作、O(n)辅助空间。编辑距离使用单位插入/删除/替换成本；计数保留任意精度整数，输出位数影响实际代价。

### 展开理解

**LCS 为什么对**：看两个前缀的末字符。若相等（比如都是 'c'），那么一定存在一个最优解同时选中这两个 'c' 配成一对——如果不是这样，我们可以把这个公共字符补进去，长度只会更长，矛盾；于是答案就是"两边各退一格的答案加一"。若不等，两个末字符不可能同时进入公共子序列（它们值不同，进去了就不相等了），所以至少丢掉一端，两种丢法里挑结果大的必然覆盖最优。这两种情况把所有可能性都分完了，且每种情况算出的值都不超过真实答案、又能构造出达到该值的方案，所以递推出的就是精确答案。

**编辑距离为什么对**：把"a 前 i 个改成 b 前 j 个"的任何一串操作，看它的最后一步。最后一步要么作用于 a 的末字符（删除它，或替换它），要么在结尾新添一个字符（必然添的是 b 的末字符）。三种可能各自对应一个更小的子问题，任何操作序列必居其一，所以三者取最小就是全局最小。相等时"对齐不花钱"之所以最优，是因为任何不含对齐的方案都可以改造成含对齐且不更贵的方案。

**计数为什么对**：所有匹配方案按"s 的最后一个被用掉的字符是哪个位置"分类。逐个字符扫 s 时，dp[j] 更新恰好发生在"x 可以充当 t 第 j 位"的时刻，叠加的是"此前已匹配好前 j−1 位"的旧方案数。倒序更新保证每个 x 只在本轮把旧值搬运一次，不重复、不遗漏。空串边界 dp[0]=1 是归纳的起点：它保证第一次命中时计数从 1 开始翻倍。

**复杂度**：三个函数都是 m×n 网格上每格常数操作，时间 O(mn)、辅助空间 O(n)（都压成了一行或一行加备份）。比如两个各 1000 字符的串是 10⁶ 格，一百万次操作瞬间完成；两个各 10⁵ 字符的串是 10¹⁰ 格，那就要换更聪明的算法了。num_distinct 的计数是任意精度整数，方案数本身可能大到有几百位，加法的实际开销会随位数增长（“正确性与复杂度”部分 提到的位数代价）。

## 8. 单元测试：每个用例在检查什么

```python
assert lcs('abcde', 'ace') == 3 and edit_distance('horse', 'ros') == 3
```

- 两条正常值测试：LCS 用 'ace' 长度 3（可对照第三节表格）；编辑距离是 LeetCode 72 的经典样例（替换 h→r、删 r、删 e 共 3 步），验证三路 min 转移写全了。

```python
assert num_distinct('rabbbit', 'rabbit') == 3 and num_distinct('abc', '') == 1
```

- 第一条是重复字符用例：三个 b 选二个，3 种组合对应 3 个方案，检验计数没有因字符相同而漏数或重数。
- 第二条是空目标边界：t 为空串时方案数是 1（唯一的方式是什么都不选），检验 dp[0]=1 这个地基。

```python
for a in words:
    for b in words:
        assert lcs(a, b) == max(map(len, subs(a) & subs(b)))
        assert edit_distance(a, b) == edit_ref(a, b)
        assert num_distinct(a, b) == sum((''.join((a[i] for i in ids)) == b for ids in combinations(range(len(a)), len(b))))
```

- 全量对拍：words 是长度 0 到 4 的全部 'a'/'b' 字符串（31 个），两两配对 961 组。LCS 与"两个子序列集合的交集中最长者"对照；编辑距离与记忆化递归参考实现 `edit_ref`（按删/插/改三分支递归）对照；num_distinct 与"枚举 s 的所有长度等于 len(b) 的下标组合、数出恰好拼成 b 的个数"对照。含空串、全同字符等极端输入，覆盖非常扎实。

In [5]:
assert lcs('abcde', 'ace') == 3 and edit_distance('horse', 'ros') == 3

assert num_distinct('rabbbit', 'rabbit') == 3 and num_distinct('abc', '') == 1

from itertools import product, combinations

from functools import cache

words = [''.join(p) for n in range(5) for p in product('ab', repeat=n)]

def subs(s):
    return {''.join((s[i] for i in ids)) for k in range(len(s) + 1) for ids in combinations(range(len(s)), k)}

@cache
def edit_ref(a, b):
    if not a or not b:
        return len(a) + len(b)
    return min(edit_ref(a[1:], b) + 1, edit_ref(a, b[1:]) + 1, edit_ref(a[1:], b[1:]) + (a[0] != b[0]))

for a in words:
    for b in words:
        assert lcs(a, b) == max(map(len, subs(a) & subs(b)))
        assert edit_distance(a, b) == edit_ref(a, b)
        assert num_distinct(a, b) == sum((''.join((a[i] for i in ids)) == b for ids in combinations(range(len(a)), len(b))))

print('N104: 所有本章断言通过')

N104: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 恢复一条LCS。

**练习 2：** 解释不同子序列对相同字符仍可能有多种索引选择。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（恢复一条 LCS）**：提示——目前只返回长度，要拿序列就得知道每个格子是从哪个邻居（左上、上、左）推来的。两种做法任选：一是把整张二维 dp 存下来，从右下角回溯——相等时选左上加一（当前字符入列），不等时走向较大的一边；二是仿照 N103 的 parent 数组，在滚动时额外记录每格的来源。用 `'abcde'`/`'ace'` 先手画出 6×4 表格再回溯，注意答案可能不唯一（'ace' 是唯一解，但别的输入如 `'abcb'`/`'bcb'` 会有多条），说明你的函数只需返回任意一条。
- **练习 2（相同字符的多种索引选择）**：提示——核心是区分"字符的值相同"和"使用的是不同的下标"。用 `'aab'`/`'ab'` 手算（答案 2）：两个 'a' 值相同，但"用下标 0 的 a + 下标 2 的 b"和"用下标 1 的 a + 下标 2 的 b"是两个不同方案。再想极端例子 `'aaa'`/`'a'`（答案 3）和 `'rabbbit'`/`'rabbit'`（答案 3），对照 num_distinct 的 dp 数组每轮怎么翻倍，解释为什么"不用 x"的那批方案永远保留在 dp[j] 里、而"用 x"的那批每次只叠加一轮。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def lcs(a, b):
    dp = [0] * (len(b) + 1)
    for x in a:
        old = dp
        dp = [0] * (len(b) + 1)
        for j, y in enumerate(b, 1):
            dp[j] = old[j - 1] + 1 if x == y else max(old[j], dp[j - 1])
    return dp[-1]

def edit_distance(a, b):
    dp = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        old = dp
        dp = [i] + [0] * len(b)
        for j, y in enumerate(b, 1):
            dp[j] = old[j - 1] if x == y else 1 + min(old[j], dp[j - 1], old[j - 1])
    return dp[-1]

def num_distinct(s, t):
    dp = [1] + [0] * len(t)
    for x in s:
        for j in range(len(t), 0, -1):
            if x == t[j - 1]:
                dp[j] += dp[j - 1]
    return dp[-1]

# 示例与回归测试
assert lcs('abcde', 'ace') == 3 and edit_distance('horse', 'ros') == 3

assert num_distinct('rabbbit', 'rabbit') == 3 and num_distinct('abc', '') == 1

from itertools import product, combinations

from functools import cache

words = [''.join(p) for n in range(5) for p in product('ab', repeat=n)]

def subs(s):
    return {''.join((s[i] for i in ids)) for k in range(len(s) + 1) for ids in combinations(range(len(s)), k)}

@cache
def edit_ref(a, b):
    if not a or not b:
        return len(a) + len(b)
    return min(edit_ref(a[1:], b) + 1, edit_ref(a, b[1:]) + 1, edit_ref(a[1:], b[1:]) + (a[0] != b[0]))

for a in words:
    for b in words:
        assert lcs(a, b) == max(map(len, subs(a) & subs(b)))
        assert edit_distance(a, b) == edit_ref(a, b)
        assert num_distinct(a, b) == sum((''.join((a[i] for i in ids)) == b for ids in combinations(range(len(a)), len(b))))

print('N104: 所有本章断言通过')

N104: 所有本章断言通过


## 11. 代表题与迁移

- **1143. Longest Common Subsequence（最长公共子序列）**：二维前缀状态 + max 转移的原型题，对应 `lcs`。
- **72. Edit Distance（编辑距离）**：按最后一次操作分解 + min 转移的原型题，对应 `edit_distance`；这道题也是"双序列 DP"最著名的代表。
- **115. Distinct Subsequences（不同的子序列）**：双序列框架下的方案数计数 + 逆序一维滚动，对应 `num_distinct`，练"max/min/sum 三种代数不能混用"这个意识。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。